# Ejercicio 10: Re-ranking

Objetivo: Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

> **Versión revisada.** Cambios respecto a la versión original:
> - **Parte 2:** BM25 vectorizado sobre la matriz dispersa (CSC) en vez de densificar columnas → mismos scores, mucho más rápido.
> - **Parte 3:** lectura robusta de `title`/`text` (`None`-safe) y conteo **simétrico** de cambios en el top-10 (docs que entran *y* que salen).
> - **Parte 4:** LTR pasa de regresión *pointwise* a **LambdaMART (LightGBM, `lambdarank`)**, que es *listwise* y optimiza directamente el orden.
> - **Parte 5:** evaluación de los tres métodos sobre las mismas queries de test y el mismo pool de candidatos (comparación justa).

## Parte 1: Preparación del corpus

- Cargar el corpus (documentos/pasajes).
- Cargar las consultas (queries).
- Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

ModuleNotFoundError: No module named 'beir'

In [ ]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

In [ ]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

In [ ]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

In [ ]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

In [ ]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

In [ ]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

### Parte 2. Retrieval inicial (baseline)
- Implementar retrieval inicial con BM25
- Obtener métricas: Recall@10 nDCG@10

> Implementación vectorizada: la matriz término-documento se guarda en formato **CSC** para acceder rápido por columna (término), y por cada término solo se tocan los documentos que lo contienen (entradas no-cero) en vez de barrer todo el corpus. La parte del denominador que no depende del término se precomputa una sola vez. Los scores son idénticos a la versión densa.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from beir.retrieval.evaluation import EvaluateRetrieval

# Corpus
doc_ids = df_corpus["doc_id"].tolist()
texts   = (df_corpus["title"].fillna("") + " " + df_corpus["text"].fillna("")).tolist()

# Indice BM25 (CSC -> acceso por columna/termino eficiente)
k1, b = 1.5, 0.75
cv = CountVectorizer(lowercase=True)
tf = cv.fit_transform(texts).tocsc()

n_docs   = tf.shape[0]
doc_lens = np.asarray(tf.sum(axis=1)).ravel()
avgdl    = doc_lens.mean()
df_term  = np.diff(tf.indptr)                      # df(t) = #docs con el termino (directo del CSC)
idf      = np.log((n_docs - df_term + 0.5) / (df_term + 0.5) + 1.0)

# Parte del denominador independiente del termino: se calcula UNA sola vez
denom_const = k1 * (1 - b + b * doc_lens / avgdl)

def bm25_retrieve(query: str, top_k: int = 100):
    q_idx  = cv.transform([query]).indices         # indices de terminos de la query
    scores = np.zeros(n_docs)
    for idx in q_idx:
        start, end = tf.indptr[idx], tf.indptr[idx + 1]
        rows    = tf.indices[start:end]            # solo docs que contienen el termino
        tf_vals = tf.data[start:end]
        norm    = tf_vals + denom_const[rows] + 1e-9
        scores[rows] += idf[idx] * (tf_vals * (k1 + 1)) / norm
    top_k = min(top_k, n_docs)
    top_i = np.argpartition(scores, -top_k)[-top_k:]
    top_i = top_i[np.argsort(scores[top_i])[::-1]]
    return {doc_ids[i]: float(scores[i]) for i in top_i}

TOP_K = 100
bm25_run = {qid: bm25_retrieve(q, top_k=TOP_K)
            for qid, q in zip(df_queries["query_id"], df_queries["query"])}

ndcg, _map, recall, _ = EvaluateRetrieval.evaluate(qrels, bm25_run, [10])
print("BM25 Baseline:")
print(f"  nDCG@10   = {ndcg['NDCG@10']:.4f}")
print(f"  Recall@10 = {recall['Recall@10']:.4f}")

### Parte 3. Implementación del re-ranking cross-encoder
- Re-rankear los top-k candidatos para cada query.
- Identificar qué documentos cambian de posición en el top 10

> El conteo de cambios es **simétrico**: registra tanto los documentos que *entran* al top-10 tras el re-ranking (`bm25_rank = None`) como los que el cross-encoder *expulsa* del top-10 (`ce_rank = None`).

In [ ]:
from sentence_transformers import CrossEncoder

model_ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def doc_fields(doc_id):
    """Devuelve (title, text) de forma robusta frente a campos None/ausentes."""
    d = corpus[doc_id]
    return (d.get("title") or ""), (d.get("text") or "")

ce_run = {}
changes_log = []

for qid, cands in bm25_run.items():
    query     = queries[qid]
    cand_list = sorted(cands.items(), key=lambda x: x[1], reverse=True)[:100]

    pairs = []
    for doc_id, _ in cand_list:
        title, text = doc_fields(doc_id)
        pairs.append([query, (title + " " + text).strip()])
    ce_scores = model_ce.predict(pairs, show_progress_bar=False)

    reranked  = sorted(zip([d for d, _ in cand_list], ce_scores),
                       key=lambda x: x[1], reverse=True)
    ce_run[qid] = {doc_id: float(s) for doc_id, s in reranked}

    # --- Cambios de posicion en el top-10 (simetrico) ---
    bm25_top10 = [d for d, _ in cand_list[:10]]
    ce_top10   = [d for d, _ in reranked[:10]]
    bm25_pos   = {d: i + 1 for i, d in enumerate(bm25_top10)}
    ce_pos     = {d: i + 1 for i, d in enumerate(ce_top10)}

    for doc_id in set(bm25_top10) | set(ce_top10):
        r_bm25 = bm25_pos.get(doc_id)     # None -> no estaba en el top-10 de BM25 (entra)
        r_ce   = ce_pos.get(doc_id)       # None -> el CE lo saco del top-10 (sale)
        if r_bm25 != r_ce:
            changes_log.append({"query_id": qid, "doc_id": doc_id,
                                "bm25_rank": r_bm25, "ce_rank": r_ce})

df_changes = pd.DataFrame(changes_log)
print(f"Cambios de posicion en top-10: {len(df_changes)}")
df_changes.head(10)

### Parte 4. Implementación del re-ranking LTR (LambdaMART)
- Re-rankear los top-k candidatos para cada query.
- Identificar qué documentos cambian de posición en el top 10

> A diferencia de una regresión *pointwise* (que aproxima la etiqueta de cada documento por separado), **LambdaMART** (`LGBMRanker(objective="lambdarank")`) es un método **listwise**: optimiza directamente el orden de los documentos *dentro de cada query*, por eso requiere el tamaño de cada grupo (`group`). El split se hace **por query** (`GroupShuffleSplit`) para que no haya fuga de queries entre train y test.

In [ ]:
# Si LightGBM no esta instalado, descomenta:
# !pip install -q lightgbm
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

def get_features(qid, doc_id, bm25_score, bm25_max):
    q_toks      = set(queries[qid].lower().split())
    title, text = doc_fields(doc_id)               # robusto frente a None (definido en Parte 3)
    d_toks      = (title + " " + text).lower().split()
    d_set       = set(d_toks)
    t_set       = set(title.lower().split())
    return [
        bm25_score / (bm25_max + 1e-9),               # f1: BM25 normalizado por query
        len(d_toks),                                   # f2: longitud del documento
        len(q_toks & d_set) / (len(q_toks) + 1e-9),   # f3: cobertura de la query en el doc
        len(q_toks & t_set) / (len(q_toks) + 1e-9),   # f4: cobertura de la query en el titulo
    ]

rows_ltr = []
for qid, cands in bm25_run.items():
    bm25_max = max(cands.values())
    for doc_id, score in cands.items():
        rel   = qrels.get(qid, {}).get(doc_id, 0)
        feats = get_features(qid, doc_id, score, bm25_max)
        rows_ltr.append({"qid": qid, "doc_id": doc_id,
                         "f1": feats[0], "f2": feats[1],
                         "f3": feats[2], "f4": feats[3], "label": rel})

df_ltr = pd.DataFrame(rows_ltr)
FEATS  = ["f1", "f2", "f3", "f4"]

# Split por query: todas las filas de una query caen en el mismo lado (sin fuga)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_ltr, groups=df_ltr["qid"]))

# LambdaMART exige filas contiguas por grupo -> ordenamos por qid
df_train  = df_ltr.iloc[train_idx].sort_values("qid").reset_index(drop=True)
df_test   = df_ltr.iloc[test_idx].sort_values("qid").reset_index(drop=True)
test_qids = df_test["qid"].unique()

# Tamano de cada grupo (query) en el orden de las filas
train_group = df_train.groupby("qid", sort=False).size().to_numpy()

ltr_model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
ltr_model.fit(df_train[FEATS], df_train["label"], group=train_group)

ltr_run = {}
for qid in test_qids:
    df_q = df_test[df_test["qid"] == qid]
    pred = ltr_model.predict(df_q[FEATS])
    ltr_run[qid] = dict(zip(df_q["doc_id"], pred.tolist()))

print(f"LTR (LambdaMART) entrenado - {df_train['qid'].nunique()} queries train / "
      f"{len(test_qids)} queries test")
print("Importancia de features:", dict(zip(FEATS, ltr_model.feature_importances_)))

### Parte 5. Evaluación post re-ranking
Calcular métricas:
- nDCG@10
- MAP
- Recall@10

> Los tres métodos se evalúan sobre **exactamente las mismas queries de test** y el mismo pool de candidatos (top-100 de BM25), para que la comparación sea justa.
>
> Nota: el nDCG de BM25 aquí **no** coincide con el de la Parte 2 — allí se midió sobre *todas* las queries y aquí solo sobre el 30% de test.

In [ ]:
from beir.retrieval.evaluation import EvaluateRetrieval

# Evaluamos los tres metodos sobre las mismas queries de test
test_q     = set(test_qids)
qrels_test = {q: v for q, v in qrels.items()    if q in test_q}
bm25_test  = {q: v for q, v in bm25_run.items() if q in test_q}
ce_test    = {q: v for q, v in ce_run.items()   if q in test_q}

summary = {}
for name, run in [("BM25", bm25_test),
                  ("CrossEncoder", ce_test),
                  ("LTR (LambdaMART)", ltr_run)]:
    ndcg, _map, recall, _ = EvaluateRetrieval.evaluate(qrels_test, run, [10])
    summary[name] = {
        "nDCG@10":   ndcg["NDCG@10"],
        "MAP@10":    _map["MAP@10"],
        "Recall@10": recall["Recall@10"],
    }

df_eval = pd.DataFrame(summary).T
print(df_eval.round(4))
df_eval.round(4).style.highlight_max(axis=0, color="lightgreen")